# ⚽ FIFA World Cup 2026 Predictions

## Strategy

This notebook implements an expected-value-maximizing prediction pipeline for the WC 2026 competition. The approach:

1. **Time-decayed Elo ratings** computed from 49,000+ international matches (1872–2026)
2. **Dixon-Coles bivariate Poisson** goals model fit on the 2014–2026 international match record, with Elo difference as a feature and a low-score correction
3. **Bookmaker odds blend** — sharp consensus odds from 30+ bookies de-vigged and blended with the model for the 1X2 marginals (65% odds / 35% model)
4. **EV-optimal scoreline picker** — for every match, the scoreline maximizing expected points under the competition scoring rule (exact = 25 / goal-diff = 10 / total = 10) is selected. This is *not* the modal scoreline — it's the one whose neighborhood captures the most probability mass
5. **Monte Carlo bracket simulator** — 5,000 simulated tournaments produce the most-likely matchups for each knockout slot, accounting for joint group-stage uncertainty and FIFA's best-third-place qualification rules
6. **Ancillary lookups** for corners / cards based on WC 2018+2022 historical averages, adjusted for predicted goal mismatch and knockout/late-round bumps

Round multipliers (R16 ×2, QF ×4, SF ×8, Final ×16) make knockout matchup picks the single highest-leverage prediction in the competition. We pick the most likely matchup from the simulator, then derive the within-match prediction from the goals model directly.

## Setup

In [1]:
import sys
import pandas as pd
import numpy as np

# Project src/ on path
sys.path.insert(0, '.')

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

In [2]:
from src.ingest import fetch_results, fetch_odds
from src.elo import fit_elo
from src.goals_model import fit_goals_model, score_distribution, lambdas
from src.odds_blend import consensus_h2h_probs, outright_probs
from src.score_optimizer import pick_score, outcome_probs, winning_team
from src.bracket_sim import aggregate_simulations
from src.submission import build_group_predictions, build_knockout_predictions
from src.team_map import resolve_playoff
import json
from pathlib import Path

## Load fixtures & raw data

In [3]:
group_fixtures = pd.read_csv('reference/data/group_fixtures.csv')
knockout_slots = pd.read_csv('reference/data/knockout_slots.csv')
print(f'Group-stage fixtures: {len(group_fixtures)}')
print(f'Knockout slots: {len(knockout_slots)}')
group_fixtures.head()

Group-stage fixtures: 72
Knockout slots: 32


,match_id,group,home_team,away_team,date_utc,venue
0,1,A,Mexico,South Africa,2026-06-11T19:00:00Z,"Estadio Azteca, Mexico City"
1,2,A,South Korea,UEFA Playoff D,2026-06-12T02:00:00Z,"Estadio Akron, Guadalajara"
2,3,B,Canada,UEFA Playoff A,2026-06-12T19:00:00Z,"BMO Field, Toronto"
3,4,D,USA,Paraguay,2026-06-13T01:00:00Z,"SoFi Stadium, Los Angeles"
4,5,D,Australia,UEFA Playoff C,2026-06-13T04:00:00Z,"BC Place, Vancouver"


In [4]:
# Pull historical results + bookmaker odds (cached after first fetch)
_ = fetch_results()
_ = fetch_odds()

## Fit Elo ratings

In [5]:
elo_path = Path('data/artifacts/elo.json')
if not elo_path.exists():
    fit_elo()
elo = json.loads(elo_path.read_text(encoding='utf-8'))
print(f'Teams rated: {len(elo)}')
print('\nTop 15 by Elo:')
for t, r in sorted(elo.items(), key=lambda x: -x[1])[:15]:
    print(f'  {t:30s} {r:7.1f}')

Teams rated: 336

Top 15 by Elo:
  Spain                           2214.4
  Argentina                       2171.3
  France                          2131.9
  England                         2077.7
  Brazil                          2042.2
  Colombia                        2039.4
  Portugal                        2028.8
  Netherlands                     2019.6
  Ecuador                         2003.6
  Germany                         1998.5
  Japan                           1989.8
  Morocco                         1987.2
  Croatia                         1979.0
  Norway                          1964.5
  Uruguay                         1958.3


## Fit Dixon-Coles goals model

In [6]:
model_path = Path('data/artifacts/goals_model.json')
if not model_path.exists():
    fixture_teams = {resolve_playoff(t) for t in set(group_fixtures['home_team']) | set(group_fixtures['away_team'])}
    fit_goals_model(fixture_teams=fixture_teams)
model = json.loads(model_path.read_text(encoding='utf-8'))
print(f"Trained on {model['n_matches']} matches (2014–2026)")
print(f"gamma (home adv) = {model['gamma']:.3f}")
print(f"theta (Elo coef) = {model['theta']:.3f}")
print(f"rho (Dixon-Coles low-score correction) = {model['rho']:.3f}")

Trained on 11580 matches (2014–2026)
gamma (home adv) = 0.239
theta (Elo coef) = 0.978
rho (Dixon-Coles low-score correction) = -0.048


## Bookmaker odds (group stage + outright winner)

In [7]:
h2h = consensus_h2h_probs()
out = outright_probs()
print(f'Group-stage matches with consensus odds: {len(h2h)} of {len(group_fixtures)}')
print(f'\nTop 10 outright winner odds (de-vigged):')
for t, p in sorted(out.items(), key=lambda x: -x[1])[:10]:
    print(f'  {t:30s} {p*100:5.2f}%')

Group-stage matches with consensus odds: 72 of 72

Top 10 outright winner odds (de-vigged):
  Spain                          16.00%
  France                         15.48%
  England                        11.71%
  Argentina                       9.14%
  Brazil                          8.73%
  Portugal                        8.00%
  Germany                         5.49%
  Netherlands                     3.56%
  Norway                          2.53%
  Belgium                         2.00%


## Group-stage predictions

For each match, we pick the (h, a) maximizing expected score-component points under the competition rule, and pick the most-likely outcome (home/draw/away) for the 40-pt group-stage winner bucket.

In [8]:
group_predictions = build_group_predictions(model, group_fixtures, h2h)
# Drop extra cols, keep only what the template expects
expected_cols = ['match_id','group','home_team','away_team','date_utc','venue',
                 'predicted_home_goals','predicted_away_goals','corners','yellow_cards','red_cards','winning_team']
group_predictions = group_predictions[expected_cols]
group_predictions

,match_id,group,home_team,away_team,date_utc,venue,predicted_home_goals,predicted_away_goals,corners,yellow_cards,red_cards,winning_team
0,1,A,Mexico,South Africa,2026-06-11T19:00:00Z,"Estadio Azteca, Mexico City",1,0,10,4,0,home
1,2,A,South Korea,UEFA Playoff D,2026-06-12T02:00:00Z,"Estadio Akron, Guadalajara",1,1,10,4,0,home
2,3,B,Canada,UEFA Playoff A,2026-06-12T19:00:00Z,"BMO Field, Toronto",1,0,10,4,0,home
3,4,D,USA,Paraguay,2026-06-13T01:00:00Z,"SoFi Stadium, Los Angeles",1,0,10,4,0,home
4,5,D,Australia,UEFA Playoff C,2026-06-13T04:00:00Z,"BC Place, Vancouver",1,1,10,4,0,away
...,...,...,...,...,...,...,...,...,...,...,...,...
67,68,L,Croatia,Ghana,2026-06-27T21:00:00Z,"Lincoln Financial Field, Philadelphia",1,1,10,4,0,home
68,69,K,Colombia,Portugal,2026-06-27T23:30:00Z,"Hard Rock Stadium, Miami",1,1,10,4,0,away
69,70,K,FIFA Playoff 1,Uzbekistan,2026-06-27T23:30:00Z,"Mercedes-Benz Stadium, Atlanta",1,0,10,4,0,home
70,71,J,Algeria,Austria,2026-06-28T02:00:00Z,"GEHA Field at Arrowhead Stadium, Kansas City",1,1,10,4,0,away


## Knockout-stage predictions

Each knockout slot's matchup is determined by which teams advance from groups. We run 5,000 Monte Carlo simulations to estimate the joint distribution of teams in each slot, then pick the most-likely matchup.

In [9]:
print('Running 5,000 tournament simulations to determine knockout matchups...')
sim_agg = aggregate_simulations(model, group_fixtures, knockout_slots, n_sims=5000, match_odds=h2h)
print('Simulation complete.')
print('\nTop 10 World Cup winner probabilities (from simulation):')
for t, c in sim_agg['final_winners'].most_common(10):
    print(f"  {t:30s} {c/sim_agg['n_sims']*100:5.1f}%")

Running 5,000 tournament simulations to determine knockout matchups...


Simulation complete.

Top 10 World Cup winner probabilities (from simulation):
  Spain                           13.4%
  Argentina                       10.8%
  Brazil                           9.4%
  England                          8.5%
  Portugal                         8.1%
  France                           7.4%
  Germany                          5.4%
  Colombia                         4.4%
  Netherlands                      4.0%
  Belgium                          4.0%


In [ ]:
knockout_predictions = build_knockout_predictions(model, knockout_slots, sim_agg, h2h)
expected_cols_k = ['match_id','round','multiplier','date_utc','venue','slot_home','slot_away',
                   'predicted_home_team','predicted_away_team',
                   'predicted_home_goals','predicted_away_goals','corners','yellow_cards','red_cards',
                   'match_winner','penalties']
knockout_predictions = knockout_predictions[expected_cols_k]
knockout_predictions

## Validation

In [11]:
# Group predictions
assert len(group_predictions) == 72, f'Expected 72 group rows, got {len(group_predictions)}'
for c in ['predicted_home_goals','predicted_away_goals','corners','yellow_cards','red_cards','winning_team']:
    nn = group_predictions[c].isna().sum()
    assert nn == 0, f'group_predictions[{c}] has {nn} NaNs'
assert set(group_predictions['winning_team']) <= {'home','away','draw'}, group_predictions['winning_team'].unique()

# Knockout predictions
assert len(knockout_predictions) == 32, f'Expected 32 knockout rows, got {len(knockout_predictions)}'
for c in ['predicted_home_team','predicted_away_team','predicted_home_goals','predicted_away_goals',
          'corners','yellow_cards','red_cards','match_winner','penalties']:
    nn = knockout_predictions[c].isna().sum()
    assert nn == 0, f'knockout_predictions[{c}] has {nn} NaNs'
assert set(knockout_predictions['match_winner']) <= {'home','away'}, knockout_predictions['match_winner'].unique()
assert set(knockout_predictions['penalties'].unique()) <= {True, False}, knockout_predictions['penalties'].unique()
print('✅ All predictions populated, schema valid.')

✅ All predictions populated, schema valid.


## Final summary

In [ ]:
print('Predicted Final:')
f = knockout_predictions[knockout_predictions['match_id'] == 104].iloc[0]
winner = f['predicted_home_team'] if f['match_winner'] == 'home' else f['predicted_away_team']
pens = ' (on penalties)' if f['penalties'] else ''
print(f"  {f['predicted_home_team']} {f['predicted_home_goals']} - {f['predicted_away_goals']} {f['predicted_away_team']}")
print(f'  Champion: {winner}{pens}')